# AstroRestore AI en Google Colab

Procedimiento inicial, pendiente de validación en Colab. Guarda los cambios en GitHub antes de ejecutarlo: se clonará la versión remota, no los archivos locales.

Selecciona un entorno con GPU. La interfaz permite subir una imagen y ejecutar una prueba de difusión imagen a imagen; la primera ejecución descarga varios GB de pesos. La prueba todavía no demuestra reducción de ruido. La disponibilidad de GPU no está garantizada.

In [ ]:
from pathlib import Path
import subprocess
import sys

proyecto = Path('/content/astro-restauracion')
if not proyecto.exists():
    subprocess.run(['git', 'clone', 'https://github.com/Sabri-Nang/astro-restauracion.git', str(proyecto)], check=True)
if not (proyecto / 'app.py').is_file():
    raise RuntimeError('La versión remota todavía no contiene app.py. Publica primero la estructura actualizada.')
subprocess.run([sys.executable, '-m', 'pip', 'install', 'uv==0.12.3'], check=True)
subprocess.run(['uv', 'sync', '--locked', '--python', '3.12'], cwd=proyecto, check=True)


In [ ]:
subprocess.run(
    ['uv', 'run', '--locked', 'python', '-c',
     "import torch; print('PyTorch:', torch.__version__); print('GPU CUDA disponible:', torch.cuda.is_available())"],
    cwd=proyecto, check=True,
)
# Esta comprobación no valida la inferencia de un modelo de difusión.


## Prueba del modelo (opcional antes de abrir la interfaz)
Sube una sola imagen PNG o TIFF de 8 bits. Se conserva la entrada y se guardan la versión preparada, la salida y sus parámetros en `outputs/prueba_colab/`. Esta celda requiere CUDA; si no está disponible, cambia el tipo de entorno. Una salida generada no prueba la eliminación de ruido ni su autenticidad científica.

In [ ]:
from google.colab import files
from IPython.display import display, Image
import json

archivos = files.upload()
if len(archivos) != 1:
    raise ValueError('Sube exactamente una imagen PNG o TIFF.')
nombre, contenido = next(iter(archivos.items()))
entrada = Path('/content') / ('entrada_prueba' + Path(nombre).suffix.lower())
entrada.write_bytes(contenido)
subprocess.run(
    ['uv', 'run', '--locked', 'python', '-m', 'src.diffusion',
     '--input', str(entrada), '--output-dir', 'outputs/prueba_colab',
     '--device', 'cuda', '--steps', '4', '--strength', '0.5', '--size', '256'],
    cwd=proyecto, check=True,
)
salidas = proyecto / 'outputs/prueba_colab'
display(Image(filename=str(salidas / 'entrada_preparada.png')))
display(Image(filename=str(salidas / 'salida.png')))
print((salidas / 'parametros.json').read_text(encoding='utf-8'))


## Abrir la interfaz
La siguiente celda mantendrá el servidor activo y solicitará un enlace público temporal de Gradio. Cualquier persona que tenga ese enlace podrá acceder mientras esté activo. Interrumpe la celda al terminar.

In [ ]:
subprocess.run(
    ['uv', 'run', '--locked', 'python', '-c',
     'from app import create_app; create_app().launch(share=True)'],
    cwd=proyecto, check=True,
)
